In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../.."))

import torch
from pytorch_implementation.perception.sparse4d.config import debug_forward_config
from pytorch_implementation.perception.sparse4d.model import Sparse4DLite

cfg = debug_forward_config(num_queries=48, decoder_layers=2)
model = Sparse4DLite(cfg).eval()

batch_size = 1
height, width = 96, 160
img = torch.randn(batch_size, cfg.num_cams, 3, height, width)

def _build_dummy_metas(batch_size, num_cams, height, width):
    projection_mat = torch.eye(4, dtype=torch.float32).view(1,1,4,4).repeat(batch_size, num_cams, 1, 1)
    for cam_idx in range(num_cams):
        projection_mat[:, cam_idx, 0, 3] = width * (0.45 + 0.02 * cam_idx)
        projection_mat[:, cam_idx, 1, 3] = height * 0.5
    image_wh = torch.tensor([width, height], dtype=torch.float32).view(1,1,2).repeat(batch_size, num_cams, 1)
    return {"projection_mat": projection_mat, "image_wh": image_wh}
img_metas = _build_dummy_metas(batch_size, cfg.num_cams, height, width)


print(f"Model: {type(model).__name__}")
print(f"Config: {cfg.name}")
print(f"Input: {tuple(img.shape)}")

In [ ]:
from typing import Any

def _first_tensor(value: Any):
    """Extract the first tensor from a nested structure."""
    import torch
    if torch.is_tensor(value):
        return value
    if isinstance(value, (tuple, list)):
        for item in value:
            t = _first_tensor(item)
            if t is not None:
                return t
    if isinstance(value, dict):
        for item in value.values():
            t = _first_tensor(item)
            if t is not None:
                return t
    return None

def _iter_tensors(value: Any):
    """Iterate over all tensors in a nested structure."""
    import torch
    if torch.is_tensor(value):
        yield value
    elif isinstance(value, (tuple, list)):
        for item in value:
            yield from _iter_tensors(item)
    elif isinstance(value, dict):
        for item in value.values():
            yield from _iter_tensors(item)

def _register_hook(module, name: str, capture: dict, handles: list) -> None:
    """Register a forward hook that stores output in capture[name]."""
    def _hook(_module, _inputs, output):
        capture[name] = output
    handles.append(module.register_forward_hook(_hook))

def _print_shape(label: str, value) -> None:
    """Print shape of first tensor in value."""
    t = _first_tensor(value)
    if t is not None:
        print(f"  {label}: {tuple(t.shape)}")
    else:
        print(f"  {label}: <no tensor>")

def _check_finite(capture: dict, outputs: dict) -> None:
    """Assert all captured intermediates and outputs are finite."""
    import torch
    for name, value in capture.items():
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in {name}"
    for name, value in outputs.items():
        if value is None:
            continue
        for t in _iter_tensors(value):
            assert torch.isfinite(t).all(), f"Non-finite in output {name}"
    print("All intermediate and final tensors are finite.")

# Sparse4D Paper-to-Code Study Guide

This note maps Sparse4D paper symbols/equations to the pure-PyTorch forward implementation in this repository.

Primary references:
- Paper: `papers/Sparse4D.pdf`
- Paper (online mirror): [Sparse4D v2: Recurrent Temporal Fusion with Sparse Model (arXiv 2305.14018)](https://arxiv.org/pdf/2305.14018.pdf)
- Reference code (online): [linxuewu/Sparse4D](https://github.com/linxuewu/Sparse4D)
- Implementation: `pytorch_implementation/perception/sparse4d/`
- Intermediate tensor tests: `tests/perception/sparse4d.py`

## 1) Canonical study setup (fixed debug run)

Use one setup so equation-to-tensor mapping stays stable across sections.

- Config:
  - `debug_forward_config(num_queries=48, decoder_layers=2)`
- Input image:
  - `img`: `[B, Ncam, C, H, W] = [1, 6, 3, 96, 160]`
- Metadata:
  - `projection_mat`: `[B, Ncam, 4, 4]`
  - `image_wh`: `[B, Ncam, 2]`

Core dimensions:
- `embed_dims = 256`
- `num_classes = 10`
- `num_decoder_layers = 2`
- `num_queries = 48`
- `box_code_size = 11`

Expected model outputs:
- `all_cls_scores`: `[L, B, Q, num_classes] = [2, 1, 48, 10]`
- `all_bbox_preds`: `[L, B, Q, 11] = [2, 1, 48, 11]`

These are verified in `tests/perception/sparse4d.py`.

## 2) Symbol dictionary (paper -> code tensors)

- `I_t` (multi-view image) -> `img`
- `F_t` (multi-level image features) -> `mlvl_feats`
- `Q_t` (instance query features) -> `instance_feature`
- `A_t` (instance anchors) -> `anchors`
- `E(A_t)` (anchor embedding) -> `anchor_embed`
- `H_l` (decoder hidden at layer `l`) -> `query` after `decoder.layers[l]`
- `\hat{c}_l` (class logits, layer `l`) -> `all_cls_scores[l]`
- `\hat{b}_l` (box prediction, layer `l`) -> `all_bbox_preds[l]`

Equation IDs below use `E<section>.<index>`.

---

## Chunk 0 - End-to-end forward contract

### Goal
Bind Sparse4D high-level pipeline to concrete module calls.

### Explicit equations
`(E0.1)` Forward path:

$$
F_t = \mathrm{ImageEncoder}(I_t),\;
(Q_t, A_t) = \mathrm{InstanceBank}(),\;
\hat{Y} = \mathrm{Head}(F_t, Q_t, A_t)
$$

`(E0.2)` Layer-wise outputs:

$$
\hat{Y} = \{(\hat{c}_l, \hat{b}_l)\}_{l=1}^{L}
$$

### Code mapping
- `Sparse4DLite.forward` and `Sparse4DLite.extract_img_feat` in `pytorch_implementation/perception/sparse4d/model.py`
- `Sparse4DHeadLite.forward` in `pytorch_implementation/perception/sparse4d/head.py`

### One sanity check
`tests/perception/sparse4d.py` asserts final output shapes for the debug config.

---

In [ ]:
# Chunk 0: End-to-end forward
with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
print("=== Sparse4D output shapes ===")
for key, val in outputs.items():
    if val is not None and torch.is_tensor(val):
        print(f"  {key}: {tuple(val.shape)}")

## Chunk 1 - Image feature extraction

### Goal
Map multi-camera image flattening and multi-level feature construction.

### Explicit equations
`(E1.1)` Camera-batch flattening:

$$
I_t \in \mathbb{R}^{B\times N_{cam}\times 3\times H\times W}
\rightarrow
I'_t \in \mathbb{R}^{(B\cdot N_{cam})\times 3\times H\times W}
$$

`(E1.2)` Multi-level features with camera reshape:

$$
F'_t{}^{(k)} \in \mathbb{R}^{(B\cdot N_{cam})\times C\times H_k\times W_k}
\rightarrow
F_t^{(k)} \in \mathbb{R}^{B\times N_{cam}\times C\times H_k\times W_k}
$$

### Code mapping
- `BackboneNeck` in `pytorch_implementation/perception/sparse4d/backbone_neck.py`
- `extract_img_feat` in `pytorch_implementation/perception/sparse4d/model.py`

### One sanity check
Tests validate `backbone.stage*` and `neck.output*` tensor shapes at each level.

---

In [ ]:
# Chunk 1: Image feature extraction
# Source: pytorch_implementation/perception/sparse4d/backbone_neck.py, model.py

capture, handles = {}, []
_register_hook(model.backbone_neck.backbone.stem, "backbone.stem", capture, handles)
for idx, stage in enumerate(model.backbone_neck.backbone.stages):
    _register_hook(stage, f"backbone.stage{idx}", capture, handles)
for idx, conv in enumerate(model.backbone_neck.neck.output_convs):
    _register_hook(conv, f"neck.output{idx}", capture, handles)

with torch.no_grad():
    img_feats = model.extract_img_feat(img)
for h in handles:
    h.remove()

print("=== Backbone/neck shapes ===")
for key in sorted(capture.keys()):
    _print_shape(key, capture[key])
print(f"\n=== Multi-level features ===")
for lvl, feat in enumerate(img_feats):
    print(f"  mlvl_feats[{lvl}]: {tuple(feat.shape)}")

## Chunk 2 - Instance bank and anchor encoding

### Goal
Connect Sparse4D sparse query initialization to concrete tensors.

### Explicit equations
`(E2.1)` Learnable sparse instance state:

$$
Q_t = \mathrm{Repeat}(Q_0, B),\;
A_t = \mathrm{Repeat}(A_0, B)
$$

`(E2.2)` Anchor encoding:

$$
E(A_t) = \mathrm{MLP}(A_t)
$$

### Code mapping
- `InstanceBankLite` in `pytorch_implementation/perception/sparse4d/instance_bank.py`
- `SparseBox3DEncoderLite` in `pytorch_implementation/perception/sparse4d/blocks.py`
- Assembly in `Sparse4DHeadLite.forward` (`pytorch_implementation/perception/sparse4d/head.py`)

### One sanity check
Tests assert shapes for `head.instance_bank` and `head.anchor_encoder`.

---

In [ ]:
# Chunk 2: Instance bank and anchor encoding
# Source: pytorch_implementation/perception/sparse4d/instance_bank.py, blocks.py

capture, handles = {}, []
_register_hook(model.head.instance_bank, "head.instance_bank", capture, handles)
_register_hook(model.head.anchor_encoder, "head.anchor_encoder", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Instance bank and anchor encoder ===")
_print_shape("instance_bank", capture["head.instance_bank"])
_print_shape("anchor_encoder", capture["head.anchor_encoder"])

## Chunk 3 - Decoder updates with image aggregation

### Goal
Map per-layer Sparse4D update blocks to the implemented decoder.

### Explicit equations
`(E3.1)` Query initialization:

$$
H_0 = Q_t + E(A_t)
$$

`(E3.2)` Decoder layer update:

$$
H_l = \mathrm{FFN}\Big(\mathrm{CrossAgg}(\mathrm{SelfAttn}(H_{l-1}, H_{l-1}, H_{l-1}), F_t)\Big)
$$

### Code mapping
- `SparseDecoderLayerLite` in `pytorch_implementation/perception/sparse4d/blocks.py`
- `Sparse4DDecoderLite` loop in `pytorch_implementation/perception/sparse4d/decoder.py`
- `DeformableFeatureAggregationLite` as a pure-PyTorch image-context surrogate in `pytorch_implementation/perception/sparse4d/blocks.py`

### One sanity check
Tests verify `decoder.layer*`, `self_attn`, `cross_attn`, and `ffn` output shapes.

---

In [ ]:
# Chunk 3: Decoder updates with image aggregation
# Source: pytorch_implementation/perception/sparse4d/decoder.py, blocks.py

capture, handles = {}, []
for idx in range(cfg.num_decoder_layers):
    dec = model.head.decoder.layers[idx]
    _register_hook(dec, f"decoder.layer{idx}", capture, handles)
    _register_hook(dec.self_attn, f"decoder.layer{idx}.self_attn", capture, handles)
    _register_hook(dec.cross_attn, f"decoder.layer{idx}.cross_attn", capture, handles)
    _register_hook(dec.ffn, f"decoder.layer{idx}.ffn", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Decoder layer shapes ===")
for idx in range(cfg.num_decoder_layers):
    print(f"\n  --- Layer {idx} ---")
    _print_shape(f"  self_attn", capture[f"decoder.layer{idx}.self_attn"])
    _print_shape(f"  cross_attn", capture[f"decoder.layer{idx}.cross_attn"])
    _print_shape(f"  ffn", capture[f"decoder.layer{idx}.ffn"])
    _print_shape(f"  layer_out", capture[f"decoder.layer{idx}"])

## Chunk 4 - Layer-wise class/box refinement and decode

### Goal
Map query states to class logits, box refinement, and final top-k decode.

### Explicit equations
`(E4.1)` Per-layer predictions:

$$
\hat{c}_l = f_{cls}(H_l),\;
\Delta b_l = f_{reg}(H_l),\;
\hat{b}_l = A_{l-1} + \Delta b_l
$$

`(E4.2)` Iterative anchor update:

$$
A_l = \mathrm{stopgrad}(\hat{b}_l)
$$

`(E4.3)` NMS-free top-k decode:

$$
(\mathrm{score}, \mathrm{label}, \mathrm{box}) =
\mathrm{TopK}(\sigma(\hat{c}_L), \hat{b}_L)
$$

### Code mapping
- `SparseBox3DRefinementLite` in `pytorch_implementation/perception/sparse4d/blocks.py`
- Iterative updates in `Sparse4DDecoderLite.forward` (`pytorch_implementation/perception/sparse4d/decoder.py`)
- `SparseBox3DDecoderLite.decode` in `pytorch_implementation/perception/sparse4d/decoder.py`
- `Sparse4DHeadLite.get_bboxes` in `pytorch_implementation/perception/sparse4d/head.py`

### One sanity check
Tests assert head branch output shapes and finite values for all captured intermediates/final tensors.

---

In [ ]:
# Chunk 4: Layer-wise class/box refinement and decode
# Source: pytorch_implementation/perception/sparse4d/blocks.py, decoder.py

capture, handles = {}, []
for idx in range(cfg.num_decoder_layers):
    _register_hook(model.head.decoder.refine_layers[idx].cls_branch,
                   f"decoder.refine{idx}.cls_branch", capture, handles)
    _register_hook(model.head.decoder.refine_layers[idx].reg_branch,
                   f"decoder.refine{idx}.reg_branch", capture, handles)

with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()

print("=== Refinement branch shapes ===")
for idx in range(cfg.num_decoder_layers):
    _print_shape(f"refine{idx}.cls_branch", capture[f"decoder.refine{idx}.cls_branch"])
    _print_shape(f"refine{idx}.reg_branch", capture[f"decoder.refine{idx}.reg_branch"])

print(f"\n=== Stacked outputs ===")
print(f"  all_cls_scores: {tuple(outputs['all_cls_scores'].shape)}")
print(f"  all_bbox_preds: {tuple(outputs['all_bbox_preds'].shape)}")

with torch.no_grad():
    decoded_out = model(img, img_metas, decode=True)
decoded = decoded_out["decoded"]
print(f"\n=== Decoded (sample 0) ===")
for key, val in decoded[0].items():
    if torch.is_tensor(val):
        print(f"  {key}: shape={tuple(val.shape)}, dtype={val.dtype}")

## 3) Dataflow diagram

```mermaid
flowchart LR
    imgInput["MultiCameraImage BxNcamx3xHxW"] --> backboneNeck[BackboneNeck]
    backboneNeck --> mlvlFeats["mlvl_feats multi-level features"]
    mlvlFeats --> featProj["Feature projection per level"]
    featProj --> projFeats["projected features for sampling"]
    instanceBank["InstanceBank anchors + features"] --> anchorEnc[AnchorEncoder]
    anchorEnc --> anchorEmbed["anchor_embed BxQxC"]
    instanceBank --> queryInit["instance_feature BxQxC"]
    queryInit --> decoder["Sparse4DDecoder L layers"]
    anchorEmbed --> decoder
    projFeats --> decoder
    imgMetas["projection_mat image_wh"] --> decoder
    decoder --> refinedQuery["refined query + anchors"]
    refinedQuery --> clsBranch[cls_branches]
    refinedQuery --> regBranch[reg_branches]
    clsBranch --> clsScores["all_cls_scores LxBxQxCcls"]
    regBranch --> bboxPreds["all_bbox_preds LxBxQx11"]
    clsScores --> decode[SparseBox3DDecoderLite]
    bboxPreds --> decode
```

## 4) One end-to-end tensor trace

1. Start with `img [1, 6, 3, 96, 160]`.
2. Backbone+FPN returns multi-level features (e.g., 4 levels with varying spatial sizes).
3. Feature projection aligns channel dims: each level `[1, 6, 256, Hl, Wl]`.
4. Instance bank provides:
   - `anchors [1, 48, 11]` (3D anchor parameters: center, size, rotation, velocity)
   - `instance_feature [1, 48, 256]` (query features).
5. Anchor encoder embeds anchors: `anchor_embed [1, 48, 256]`.
6. Run 2 decoder layers, each consisting of:
   - Self-attention among queries: `[1, 48, 256]`.
   - Deformable feature aggregation (sample from multi-view multi-level features).
   - FFN refinement.
   - Anchor refinement: update `anchors [1, 48, 11]`.
7. Per-layer head branches:
   - `all_cls_scores [2, 1, 48, 10]`
   - `all_bbox_preds [2, 1, 48, 11]`.
8. Box decoder converts anchor codes to metric 3D boxes, top-k selection.

## 5) Study drills (self-check questions)

1. Why does Sparse4D use explicit 3D anchors rather than learned reference points like DETR?
2. What concrete tensors correspond to paper symbols `Q_t`, `A_t`, and `E(A_t)`?
3. How does `AnchorEncoder` convert an 11-D anchor to an embedding — what is the architecture?
4. What is "deformable feature aggregation" and how does it differ from standard cross-attention?
5. Why does anchor refinement happen at every decoder layer rather than only at the end?
6. How does `projection_mat` connect 3D anchor locations to 2D image sampling points?
7. What is the role of `image_wh` in the sampling process?
8. Why is `box_code_size = 11` — what are the 11 components?
9. How does the instance bank persist across frames in the temporal variant?
10. What would happen if you removed anchor refinement and kept anchors fixed?

## 6) Practical reading order for this note

1. Read Sections 1 and 2 once.
2. Walk through Chunk 1 (backbone and anchor encoding) — understand inputs.
3. Study Chunk 2 (decoder with self-attention and deformable aggregation).
4. Study Chunk 3 (anchor refinement per layer).
5. Study Chunk 4 (detection heads and box decode).
6. Re-read Chunk 0 (end-to-end) to tie the full pipeline together.
7. Re-run the tensor trace in Section 4 while stepping through code.
8. Answer study drills without looking at code, then verify.

## 7) Strict parity notes and pure-PyTorch replacements

- Behavioral parity is pinned to frozen Sparse4D anchors in `study/markdown/strict_parity_anchor_manifest.md`.
- Instance-bank temporal update/cache semantics and per-layer anchor refinement are preserved in strict mode.
- Projection-aware feature sampling is implemented with pure PyTorch gather/matmul/grid sampling operators.
- Decode keeps Sparse4D top-k ordering and box denormalization semantics.

In [ ]:
# Final finiteness validation
capture, handles = {}, []
_register_hook(model.backbone_neck.backbone.stem, "backbone.stem", capture, handles)
for idx, stage in enumerate(model.backbone_neck.backbone.stages):
    _register_hook(stage, f"backbone.stage{idx}", capture, handles)
with torch.no_grad():
    outputs = model(img, img_metas, decode=False)
for h in handles:
    h.remove()
_check_finite(capture, outputs)